<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/946_EAASv3_TrendAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EaaS v3 Trend Analysis & Trigger History — Code Review

## What This Module Does

The purpose of this module is to transform evaluation results into **historical operational intelligence**.

After each evaluation run, the system needs to answer questions like:

* Are problems appearing for the first time?
* Are known issues persisting across releases?
* Are previous problems being resolved?
* Is the system trending toward risk or stability?

This module records that information in the **trigger history**, allowing the evaluation system to track the long-term health of the orchestrator.

Without this capability, evaluation would be a **static scorecard**.

With it, the system becomes **AI reliability monitoring**.

---

# Where This Fits in the Agent Architecture

Within the orchestrator pipeline, this module sits **between scoring and reporting**.

Pipeline flow:

```
Data Loading
      ↓
Scenario Scoring
      ↓
Trend Analysis & Trigger History
      ↓
Executive Report
```

The scoring engine calculates metrics.

This module interprets those metrics and answers a different question:

> **Is the system getting better or worse?**

That distinction is critical for production AI systems.

---

# Time-Stamped Monitoring

## `_now_utc_iso()`

This helper generates a **standardized UTC timestamp**.

```python
datetime.now(timezone.utc).isoformat()
```

Using UTC timestamps prevents common operational issues:

* time zone confusion
* inconsistent logging
* difficult historical analysis

This small design choice ensures that the system’s historical data remains **globally consistent and audit-friendly**.

---

# System Health Classification

## `_infer_system_status`

This function converts evaluation metrics into a **high-level system status**.

Possible outputs:

```
healthy
warning
critical
```

The decision logic prioritizes **risk triggers** over raw scores.

Example logic:

| Condition                   | Result   |
| --------------------------- | -------- |
| critical risk trigger       | critical |
| regression trigger          | warning  |
| low evaluation score        | warning  |
| no failures and no triggers | healthy  |

This hierarchy reflects a realistic operational philosophy.

A system might have a high score but still be risky if **critical triggers appear**.

For leadership, this function essentially answers:

> **Should we be worried about this release?**

That’s exactly the type of signal executives want.

---

# Trigger Persistence Detection

## `_compute_trigger_counts`

This function is particularly valuable because it detects **trigger persistence across runs**.

Three metrics are computed:

| Metric                   | Meaning                                   |
| ------------------------ | ----------------------------------------- |
| trigger_count            | number of active triggers                 |
| persistent_trigger_count | triggers continuing from the previous run |
| resolved_trigger_count   | triggers that disappeared                 |

Example:

Previous run triggers:

```
policy_violation_trigger
latency_trigger
```

Current run triggers:

```
policy_violation_trigger
```

Results:

```
trigger_count = 1
persistent_trigger_count = 1
resolved_trigger_count = 1
```

This tells the system that:

* one issue remains unresolved
* one issue was fixed

That information is extremely valuable when tracking reliability across releases.

---

# Building a Trigger History Record

## `_build_trigger_history_entry`

This function converts run metrics into a **structured historical record**.

Example fields include:

```
run_id
run_date
evaluation_score
pass_rate
outcome_accuracy
high_risk_failures
human_review_rate
system_status
trigger_flags
```

Additionally, the function stores boolean indicators for each trigger type:

```
regression_trigger
critical_risk_trigger
avg_latency_trigger
policy_violation_trigger
```

This structure makes the trigger history easy to query later.

For example, you could quickly answer:

* How often did policy violations occur?
* Which releases triggered risk alerts?
* When did the system become stable?

This is exactly the type of telemetry used in reliability monitoring systems.

---

# Recording Evaluation History

## `update_trigger_history_with_run`

This is the main public function in the module.

It performs three steps:

### Step 1 — Normalize run metrics

The function converts the dictionary metrics into a `RunMetrics` object.

This ensures consistent typing and avoids schema drift.

---

### Step 2 — Calculate trigger trends

It calls:

```
_compute_trigger_counts
```

to detect:

* persistent issues
* resolved issues
* total active triggers

This gives the system historical awareness.

---

### Step 3 — Append new history entry

A new trigger history entry is created and appended to the existing history.

```
updated_history = old_history + new_entry
```

This keeps the system’s operational history growing over time.

---

# Why This Design Is Strong

Several architectural choices here are excellent.

---

## 1. Historical context is first-class

The system treats historical performance as a core evaluation dimension.

Many AI systems only measure:

```
current performance
```

Your system measures:

```
performance over time
```

This is essential for reliability engineering.

---

## 2. Risk signals are explicit

Instead of burying risk signals inside metrics, the system surfaces them clearly:

```
critical_risk_trigger
policy_violation_trigger
latency_trigger
```

This improves operational clarity.

---

## 3. Trigger persistence is tracked

Persistent issues are far more concerning than one-time anomalies.

Tracking persistence allows the system to detect:

```
chronic instability
```

rather than occasional noise.

---

## 4. System health is summarized

Executives should not have to interpret dozens of metrics.

The `system_status` field condenses evaluation results into a simple signal:

```
healthy
warning
critical
```

This is exactly the level of abstraction leadership needs.

---

# Why CEOs Would Appreciate This Design

A business leader looking at this system would notice three reassuring things.

### 1. The system monitors itself over time

This reduces the risk of silent degradation.

### 2. Problems are clearly surfaced

Triggers act as early-warning signals.

### 3. Improvements are measurable

Resolved triggers demonstrate operational progress.

Together, these capabilities turn evaluation into a **continuous governance process**.

---

# One Small Improvement Suggestion

Right now, `_compute_trigger_counts` detects previous triggers by scanning the previous entry for boolean fields.

Example logic:

```python
flag for flag, active in last_entry.items() if isinstance(active, bool) and active
```

This works but could accidentally include unrelated boolean fields in the future.

A slightly safer approach would be explicitly checking known trigger names.

For example:

```
KNOWN_TRIGGERS = [
    "critical_risk_trigger",
    "regression_trigger",
    "avg_latency_trigger",
    ...
]
```

This protects the system if additional boolean metadata is added later.

---

# Another Optional Enhancement

You could also track **long-term trigger streaks**.

Example:

```
latency_trigger_streak = 4 runs
```

This makes it easier to detect systemic issues.

But this is definitely a **v4-level improvement**, not something needed for your current version.

---

# What Stands Out Most

The most impressive part of this module is the concept of **trigger persistence tracking**.

That idea transforms the evaluation engine into something much closer to:

```
AI system reliability monitoring
```

rather than:

```
AI benchmarking
```

That’s a big conceptual leap.

---

# Final Assessment

This module is:

* architecturally clean
* operationally meaningful
* aligned with real reliability monitoring patterns

It strengthens the entire EaaS system by adding **historical awareness and risk monitoring**.

Together with your scoring engine, it forms the backbone of a **risk-aware AI evaluation platform**.


In [ ]:
"""Trend and trigger-history utilities for EaaS v3."""

from __future__ import annotations

from typing import Any, Dict, List, Tuple
from datetime import datetime, timezone

from .scoring import RunMetrics


def _now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _infer_system_status(
    evaluation_score: float,
    high_risk_failures: int,
    trigger_flags: List[str],
) -> str:
    if "critical_risk_trigger" in trigger_flags:
        return "critical"
    if "regression_trigger" in trigger_flags:
        return "warning"
    if evaluation_score < 0.75:
        return "warning"
    if high_risk_failures == 0 and not trigger_flags:
        return "healthy"
    return "warning"


def _compute_trigger_counts(
    trigger_history: List[Dict[str, Any]],
    current_flags: List[str],
) -> Dict[str, int]:
    """Compute trigger_count, persistent_trigger_count, resolved_trigger_count."""
    trigger_count = len(current_flags)

    if not trigger_history:
        return {
            "trigger_count": trigger_count,
            "persistent_trigger_count": 0,
            "resolved_trigger_count": 0,
        }

    last_entry = trigger_history[-1]
    previous_flags = set(flag for flag, active in last_entry.items() if isinstance(active, bool) and active)

    current_flag_set = set(current_flags)
    persistent = len(previous_flags & current_flag_set)
    resolved = len(previous_flags - current_flag_set)

    return {
        "trigger_count": trigger_count,
        "persistent_trigger_count": persistent,
        "resolved_trigger_count": resolved,
    }


def _build_trigger_history_entry(
    *,
    run_id: str,
    run_metrics: RunMetrics,
    scoring_version: str,
    trigger_counts: Dict[str, int],
) -> Dict[str, Any]:
    flags = set(run_metrics.trigger_flags)

    entry: Dict[str, Any] = {
        "run_id": run_id,
        "run_date": _now_utc_iso().split("T")[0],
        "target_version": "",  # optional; can be filled by caller if needed
        "regression_trigger": "regression_trigger" in flags,
        "critical_risk_trigger": "critical_risk_trigger" in flags,
        "low_evaluation_score_trigger": "low_evaluation_score_trigger" in flags,
        "avg_latency_trigger": "avg_latency_trigger" in flags,
        "p95_latency_trigger": "p95_latency_trigger" in flags,
        "high_human_review_trigger": "high_human_review_trigger" in flags,
        "hallucination_trigger": "hallucination_trigger" in flags,
        "policy_violation_trigger": "policy_violation_trigger" in flags,
        "weighted_failure_trigger": "weighted_failure_trigger" in flags,
        "trigger_count": trigger_counts["trigger_count"],
        "persistent_trigger_count": trigger_counts["persistent_trigger_count"],
        "resolved_trigger_count": trigger_counts["resolved_trigger_count"],
        "system_status": _infer_system_status(
            evaluation_score=run_metrics.evaluation_score,
            high_risk_failures=run_metrics.high_risk_failures,
            trigger_flags=run_metrics.trigger_flags,
        ),
        "evaluation_score": run_metrics.evaluation_score,
        "pass_rate": run_metrics.overall_pass_rate,
        "outcome_accuracy": run_metrics.outcome_accuracy,
        "high_risk_failures": run_metrics.high_risk_failures,
        "human_review_rate": run_metrics.human_review_rate,
        "trigger_flags": run_metrics.trigger_flags,
        "scoring_version": scoring_version,
        "notes": "",
    }
    return entry


def update_trigger_history_with_run(
    *,
    run_id: str,
    run_metrics: Dict[str, Any],
    trigger_history: List[Dict[str, Any]],
    scoring_version: str,
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:
    """Return (new_entry, updated_history) including appended trigger history."""
    metrics_obj = RunMetrics(
        run_id=run_id,
        total_scenarios=int(run_metrics.get("total_scenarios", 0)),
        scenarios_passed=int(run_metrics.get("scenarios_passed", 0)),
        overall_pass_rate=float(run_metrics.get("overall_pass_rate", 0.0)),
        issue_classification_accuracy=float(run_metrics.get("issue_classification_accuracy", 0.0)),
        resolution_path_accuracy=float(run_metrics.get("resolution_path_accuracy", 0.0)),
        outcome_accuracy=float(run_metrics.get("outcome_accuracy", 0.0)),
        high_risk_failures=int(run_metrics.get("high_risk_failures", 0)),
        human_review_rate=float(run_metrics.get("human_review_rate", 0.0)),
        avg_latency_ms=float(run_metrics.get("avg_latency_ms", 0.0)),
        p95_latency_ms=float(run_metrics.get("p95_latency_ms", 0.0)),
        hallucination_rate=float(run_metrics.get("hallucination_rate", 0.0)),
        policy_violation_rate=float(run_metrics.get("policy_violation_rate", 0.0)),
        weighted_failure_rate=float(run_metrics.get("weighted_failure_rate", 0.0)),
        evaluation_score=float(run_metrics.get("evaluation_score", 0.0)),
        regression_flags=list(run_metrics.get("regression_flags", [])),
        improvement_flags=list(run_metrics.get("improvement_flags", [])),
        trigger_flags=list(run_metrics.get("trigger_flags", [])),
    )

    counts = _compute_trigger_counts(trigger_history=trigger_history, current_flags=metrics_obj.trigger_flags)
    new_entry = _build_trigger_history_entry(
        run_id=run_id,
        run_metrics=metrics_obj,
        scoring_version=scoring_version,
        trigger_counts=counts,
    )

    updated_history = list(trigger_history) + [new_entry]
    return new_entry, updated_history

